# ADHD FC Pipeline: Hubs + Community Structure (rechecked twice, two real bugs fixed)

**Recheck 1 found:** negative-strength hubs were computed but never included in
the output text. Fixed -- now reports both POSITIVE and NEGATIVE hubs.

**Recheck 2 found:** `build_community_description` crashed (`TypeError`) if any
Yeo-7 network ends up with too few ROIs to form a valid within-network pair
(e.g. a network with only 1 ROI) -- confirmed with a live crash reproduction,
not hypothetical. Fixed -- now reports "insufficient ROIs to compute" for that
pair instead of crashing, so a sparse network surfaces as a visible note
rather than silently killing the whole run.

Fill in your real `roi_network` (Yeo-7 label per ROI) in Section 5 before
running the community-structure part.


## SECTION 1: SETUP

In [1]:
# =============================================================================
# Why: standard imports plus bctpy for the neuroscience-standard weighted
# strength calculation. N_ROIS=116 matches the AAL116 atlas used throughout
# this project.

import numpy as np
import pandas as pd
import json
import bct

N_ROIS = 116



In [2]:
import json
import os
import glob
from xml.etree import ElementTree
from sklearn.utils import Bunch
from nilearn import image as _image

AAL_LOCAL_DIR = "/home/siu856622573/Code/aal"

def load_local_aal(base_dir=AAL_LOCAL_DIR):
    """Load the AAL atlas from a local directory (instead of downloading from nilearn)."""
    nii_candidates = sorted(glob.glob(os.path.join(base_dir, "**", "*.nii"), recursive=True))
    xml_candidates = sorted(glob.glob(os.path.join(base_dir, "**", "*.xml"), recursive=True))

    nii_path = next((p for p in nii_candidates if "aal" in os.path.basename(p).lower()),
                     nii_candidates[0] if nii_candidates else None)
    xml_path = next((p for p in xml_candidates if "aal" in os.path.basename(p).lower()),
                     xml_candidates[0] if xml_candidates else None)

    if nii_path is None or xml_path is None:
        raise FileNotFoundError(
            f"Could not find AAL .nii/.xml under {base_dir}. "
            f"Found .nii: {nii_candidates}, .xml: {xml_candidates}"
        )

    labels = ["Background"]
    indices = ["0"]
    xml_tree = ElementTree.parse(xml_path)
    root = xml_tree.getroot()
    for label in root.iter("label"):
        indices.append(label.find("index").text)
        labels.append(label.find("name").text)

    return Bunch(maps=nii_path, labels=labels, indices=indices)


def build_aal_roi_names():
    """Returns a list of 116 AAL region names, ordered to match ROI_NETWORK / the FC matrix
    columns (ascending AAL label value -- same convention used in build_aal_to_yeo7)."""
    aal = load_local_aal()
    aal_img = _image.load_img(aal.maps)
    aal_data = aal_img.get_fdata()
    aal_values = np.unique(aal_data)
    aal_values = aal_values[aal_values != 0]

    label_lookup = {int(idx): lab for idx, lab in zip(aal.indices, aal.labels)}
    names = [label_lookup[int(v)] for v in aal_values]
    return names


ROI_NAMES = build_aal_roi_names()
assert len(ROI_NAMES) == N_ROIS

with open("aal_roi_names.json", "w") as f:
    json.dump(ROI_NAMES, f, indent=2)

print(f"Saved {len(ROI_NAMES)} ROI names -> aal_roi_names.json")


Saved 116 ROI names -> aal_roi_names.json


In [3]:
from nilearn import datasets
yeo = datasets.fetch_atlas_yeo_2011()
print(yeo.keys())

[fetch_atlas_yeo_2011] Dataset found in /home/siu856622573/nilearn_data/yeo_2011

dict_keys(['maps', 'labels', 'description', 'lut', 'atlas_type', 'template', 'anat'])


In [4]:
from nilearn import datasets, image
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
import numpy as np

YEO7_NAMES = {
    1: "Vis", 2: "SomMot", 3: "DorsAttn", 4: "SalVentAttn",
    5: "Limbic", 6: "Cont", 7: "Default",
}


def build_aal_to_yeo7():
    """Assign each of the 116 AAL ROIs (in .1D column order) to one of Yeo's
    7 networks, via majority-vote voxel overlap with the resampled Yeo-7 atlas."""
    aal = load_local_aal()  
    aal_img = image.load_img(aal.maps)
    aal_data = aal_img.get_fdata()
    aal_values = np.unique(aal_data)
    aal_values = aal_values[aal_values != 0]

    yeo_img = image.load_img(yeo.maps)

    yeo_resampled = image.resample_to_img(
        yeo_img,
        aal_img,
        interpolation="nearest"
    )
    yeo_data = yeo_resampled.get_fdata().squeeze()

    roi_network = []
    for val in aal_values:
        mask = aal_data == val
        vox = yeo_data[mask]
        vox = vox[vox > 0]  
        if vox.size == 0:
            roi_network.append(None)
            continue
        labels, counts = np.unique(vox, return_counts=True)
        roi_network.append(int(labels[np.argmax(counts)]))
    return roi_network


ROI_NETWORK = build_aal_to_yeo7()
assert len(ROI_NETWORK) == N_ROIS

NETWORKS = sorted(set(n for n in ROI_NETWORK if n is not None))

print("ROI counts per Yeo-7 network:")

print(pd.Series([YEO7_NAMES.get(n, "Unassigned") for n in ROI_NETWORK]).value_counts())


# --- ADDED: string-label version for the hub/community-structure pipeline ---
# That pipeline's compute_network_pair_fc expects roi_network as a list of
# NAME strings ("Vis", "Default", ...), not the integer Yeo labels (1-7)
# ROI_NETWORK stores. Convert here, once, so both pipelines can share the
# same underlying assignment without duplicating the atlas-overlap step.
roi_network = [YEO7_NAMES.get(n, "Unassigned") for n in ROI_NETWORK]

n_unassigned = roi_network.count("Unassigned")
if n_unassigned > 0:
    print(f"\nWARNING: {n_unassigned} ROI(s) had no Yeo-7 overlap and are labeled 'Unassigned'.")
    print("These ROIs are automatically EXCLUDED from all network-pair averages below")
    print("(compute_network_pair_fc only groups ROIs matching a name in NETWORK_NAMES,")
    print("and 'Unassigned' is deliberately left out of that list) -- not treated as a")
    print("real 8th network. If this count is large, double-check AAL/Yeo atlas alignment")
    print("before trusting the resulting network-pair values.")

# NETWORK_NAMES excludes "Unassigned" on purpose -- see warning above.
NETWORK_NAMES = sorted(set(roi_network))
print(f"\nNETWORK_NAMES for community-structure pipeline: {NETWORK_NAMES}")

ROI counts per Yeo-7 network:
Unassigned     22
Vis            22
Limbic         20
Default        19
SomMot         13
Cont           10
SalVentAttn     8
DorsAttn        2
Name: count, dtype: int64

These ROIs are automatically EXCLUDED from all network-pair averages below
(compute_network_pair_fc only groups ROIs matching a name in NETWORK_NAMES,
and 'Unassigned' is deliberately left out of that list) -- not treated as a
real 8th network. If this count is large, double-check AAL/Yeo atlas alignment
before trusting the resulting network-pair values.

NETWORK_NAMES for community-structure pipeline: ['Cont', 'Default', 'DorsAttn', 'Limbic', 'SalVentAttn', 'SomMot', 'Unassigned', 'Vis']


In [5]:
with open("roi_network.json", "w") as f:
    json.dump(roi_network, f, indent=2)

json.dump(NETWORK_NAMES, open("network_names.json", "w"))

In [7]:
import json
import re
import numpy as np
import bct

# =============================================================================
# Load exactly what you asked: the saved ROI names + the named top-10% edge
# file. No npz, no matrix reconstruction -- everything below is derived
# straight from these two files.
# =============================================================================
N_ROIS = 116
HUB_PERCENTILE = 80   # top 20% within-subject
ROI_NAMES_JSON = "aal_roi_names.json"
EDGE_LIST_NAMES = "/home/siu856622573/Code/edge_list_top10pos_names.json"
DATASET_NAME = "ADHD"
PREPROCESS_TEMPLATE = "AAL116"
META_CSV = "subject_summary.csv"


with open(ROI_NAMES_JSON) as f:       # "aal_roi_names.json", already defined in your notebook
    roi_names = json.load(f)
assert len(roi_names) == N_ROIS

with open(EDGE_LIST_NAMES) as f:      # "edge_list_top10pos_names.json", already defined in your notebook
    edge_list_names_all = json.load(f)

if os.path.exists(META_CSV):
    meta = pd.read_csv(META_CSV, dtype={"subject_id": str})
else:
    meta = None
# roi_network: your existing Yeo-7 string-label list, length 116 -- fill in
# from your build_aal_to_yeo7() cell before running the community section.
# roi_network = [YEO7_NAMES.get(n, "Unassigned") for n in ROI_NETWORK]
# NETWORK_NAMES = sorted(set(roi_network) - {"Unassigned"})


# =============================================================================
# Step 1: parse ONLY the Edge feature section -- Node feature section
# (positive-only means) is discarded entirely, so hub/community are derived
# purely from edges, not leaked from the pre-computed node means.
# =============================================================================

def parse_edge_section_only(text):
    """text = the full saved string ('Node feature:\\n...\\n\\nEdge feature:\\n...').
    Returns just the edge lines; the Node feature section is never read."""
    edge_block = text.split("Edge feature:\n", 1)[1]
    return [line for line in edge_block.strip().splitlines() if line]


def parse_named_edge_line(line):
    """'RegionA:weight:RegionB' -> (name_i, weight, name_j).
    Defensive check: this pipeline requires positive-only edges everywhere --
    verify it here rather than silently trusting the input file."""
    name_i, w, name_j = line.split(":")
    w = float(w)
    assert w > 0, f"Non-positive edge weight found ({w}) -- input must be positive-only (FC > 0)."
    return (name_i, w, name_j)


def edges_for_subject(subject_id):
    """Raw (name_i, weight, name_j) tuples for one subject, weight is the
    RAW connectivity strength (not a z-score) -- matches EDGE_LIST_NAMES."""
    text = edge_list_names_all[subject_id]
    edge_lines = parse_edge_section_only(text)
    return [parse_named_edge_line(line) for line in edge_lines]


_name_to_idx = {name: i for i, name in enumerate(roi_names)}   # 0-indexed lookup


# =============================================================================
# Step 2: hub ground truth -- binarize the edge set (present/absent, ignore
# magnitude), bct.degrees_und for degree count, within-subject top-20%,
# cohort z-score of degree across subjects.
# =============================================================================

def binary_adjacency_from_named_edges(edges, name_to_idx, n_rois):
    A = np.zeros((n_rois, n_rois))
    for (name_i, w, name_j) in edges:
        i, j = name_to_idx[name_i], name_to_idx[name_j]
        A[i, j] = 1
        A[j, i] = 1
    return A

def compute_degree_hubs_from_json(edge_list_names_all, roi_names, top_n=20):
    subject_ids = list(edge_list_names_all.keys())
    n_subjects = len(subject_ids)
    degrees_all = np.zeros((n_subjects, N_ROIS))

    # Step 1: degree for every ROI, every subject
    for s, sid in enumerate(subject_ids):
        edges = edges_for_subject(sid)
        A = binary_adjacency_from_named_edges(edges, _name_to_idx, N_ROIS)
        degrees_all[s] = bct.degrees_und(A)

    degree_z = (degrees_all - degrees_all.mean(axis=0)) / (degrees_all.std(axis=0) + 1e-8)

    hub_results = {}
    for s, sid in enumerate(subject_ids):
        # Step 2: sort all 116 ROIs by degree, descending - big -> small
        # Step 3: keep top_n (20) -- ties broken by |z| so it's still deterministic
        # degrees_all[s] → sorts - ascending; -degrees_all[s] → sorts descending
        # Tie-breaker: If degrees are equal, higher absolute z-score first (-np.abs(degree_z[s])).
        order = np.lexsort((-np.abs(degree_z[s]), -degrees_all[s]))
        hub_idx = order[:top_n]

        hub_results[sid] = sorted(
            [
                {"roi": roi_names[i], "degree": int(degrees_all[s, i]), "cohort_z": float(degree_z[s, i])}
                for i in hub_idx
            ],
            key=lambda x: x["cohort_z"], reverse=True
        )
    return hub_results, subject_ids


hub_results, subject_ids = compute_degree_hubs_from_json(edge_list_names_all, roi_names, top_n=20)


# =============================================================================
# Step 3: community ground truth -- within/between Yeo-7 network EDGE
# DENSITY (fraction of possible pairs present in the binarized graph),
# using the SAME parsed edges as the hub computation above -- weight is
# discarded here, only presence/absence matters, same as hub.
# =============================================================================

def compute_network_pair_fc_named(edges, roi_network, network_names, name_to_idx):
    """BINARIZED version -- consistent with the hub computation. Instead of
    averaging raw FC weights (which would apply DIFFERENT preprocessing to
    community vs. hub, violating the 'same preprocessing everywhere' rule),
    this reports EDGE DENSITY: what fraction of all possible ROI-pairs
    within/between a network pair are actually present (=1) in the
    binarized top-10% graph. A pair with zero possible ROI-pairs (e.g. a
    network with only 1 ROI) is None (insufficient data), not 0."""
    assert len(roi_network) == N_ROIS, f"roi_network has {len(roi_network)} entries, expected {N_ROIS}."

    # Set of (a, b) 0-indexed pairs that are "1" in the binarized graph --
    # presence only, weight discarded, matching the hub binarization exactly.
    present_pairs = set()
    for (name_i, w, name_j) in edges:
        a, b = name_to_idx[name_i], name_to_idx[name_j]
        present_pairs.add((min(a, b), max(a, b)))

    within, between = {}, {}
    for a_idx, net_a in enumerate(network_names):
        idx_a = [k for k, n in enumerate(roi_network) if n == net_a]
        for net_b in network_names[a_idx:]:
            idx_b = [k for k, n in enumerate(roi_network) if n == net_b]

            if net_a == net_b:
                total_possible = len(idx_a) * (len(idx_a) - 1) // 2
                if total_possible == 0:
                    within[net_a] = None
                    continue
                n_present = sum(
                    1 for x in range(len(idx_a)) for y in range(x + 1, len(idx_a))
                    if (min(idx_a[x], idx_a[y]), max(idx_a[x], idx_a[y])) in present_pairs
                )
                within[net_a] = n_present / total_possible
            else:
                total_possible = len(idx_a) * len(idx_b)
                if total_possible == 0:
                    between[f"{net_a}-{net_b}"] = None
                    continue
                n_present = sum(
                    1 for a in idx_a for b in idx_b
                    if (min(a, b), max(a, b)) in present_pairs
                )
                between[f"{net_a}-{net_b}"] = n_present / total_possible
        within = dict(sorted(within.items(), key=lambda x: x[1], reverse=True))

        between = dict(sorted(between.items(), key=lambda x: x[1], reverse=True))
    return {"within_network": within, "between_network": between}

# =============================================================================
# Step 5: run everything, save ground truth + prompts separately.
# =============================================================================

def run_all_from_json(edge_list_names_all, roi_names, roi_network, network_names):
    hub_results, subject_ids = compute_degree_hubs_from_json(edge_list_names_all, roi_names)

    ground_truth = {}
    for sid in subject_ids:
        edges = edges_for_subject(sid)
        community = compute_network_pair_fc_named(edges, roi_network, network_names, _name_to_idx)
        ground_truth[sid] = {
            "hub_regions": hub_results[sid],
            "within_network": community["within_network"],
            "between_network": community["between_network"],
        }

    return ground_truth


# =============================================================================
# Save the edge-only text per subject -- the SAME parsed edges used for hub
# and community above, saved separately so it can be loaded directly as
# LLM input without re-parsing the combined Node+Edge file each time.
# =============================================================================

def save_edge_only_file(edge_list_names_all, out_path="edge_only_from_top10pos_names.json"):
    edge_only = {
        sid: "\n".join(parse_edge_section_only(text))
        for sid, text in edge_list_names_all.items()
    }
    with open(out_path, "w") as f:
        json.dump(edge_only, f, indent=2)
    print(f"Saved edge-only text for {len(edge_only)} subjects -> {out_path}")
    return edge_only


# --- Run (fill in roi_network first) ---
ground_truth = run_all_from_json(edge_list_names_all, roi_names, roi_network, NETWORK_NAMES)
with open("hub_community_ground_truth.json", "w") as f:
    json.dump(ground_truth, f, indent=2)

edge_only_texts = save_edge_only_file(edge_list_names_all)
# -> load this directly later as: brain_graph_texts = json.load(open("edge_only_from_top10pos_names.json"))


Saved edge-only text for 768 subjects -> edge_only_from_top10pos_names.json


In [8]:
import json

with open("hub_community_ground_truth.json") as f:
    ground_truth = json.load(f)

# Split into two separate {subject_id: value} dicts -- same flat shape as
# structure_prompts_all, so both load and use exactly like:
#   hub_ground_truth = dict(json.load(open("hub_ground_truth.json")))

hub_ground_truth = {
    sid: record["hub_regions"]
    for sid, record in ground_truth.items()
}

community_ground_truth = {
    sid: {
        "within_network": record["within_network"],
        "between_network": record["between_network"],
    }
    for sid, record in ground_truth.items()
}

with open("hub_ground_truth_20roi.json", "w") as f:
    json.dump(hub_ground_truth, f, indent=2)

with open("community_ground_truth.json", "w") as f:
    json.dump(community_ground_truth, f, indent=2)

print(f"Saved hub_ground_truth_20roi.json ({len(hub_ground_truth)} subjects)")
print(f"Saved community_ground_truth.json ({len(community_ground_truth)} subjects)")


Saved hub_ground_truth_20roi.json (768 subjects)
Saved community_ground_truth.json (768 subjects)


In [9]:
import json
import os

# Define the file path
file_path = 'hub_ground_truth_20roi.json'

def check_rois(path, target_count=20):
    if not os.path.exists(path):
        print(f"Error: File not found at {path}")
        return

    with open(path, 'r') as f:
        data = json.load(f)

    all_match = True
    mismatched_subjects = []

    print(f"Checking subjects in {path}...\n")

    for subject_id, rois in data.items():
        count = len(rois)
        if count != target_count:
            all_match = False
            mismatched_subjects.append((subject_id, count))
    
    if all_match:
        print(f"Success! Every subject ({len(data)} total) has exactly {target_count} ROIs.")
    else:
        print(f"Alert: Found {len(mismatched_subjects)} subjects that do NOT have {target_count} ROIs:")
        for sub_id, count in mismatched_subjects:
            print(f" - Subject {sub_id}: {count} ROIs")

# Run the check
check_rois(file_path)

Checking subjects in hub_ground_truth_20roi.json...

Success! Every subject (768 total) has exactly 20 ROIs.


## SECTION 6: COMMUNITY GROUND TRUTH -- WITHIN/BETWEEN NETWORK COUNTS

Count-based version (not density). For every edge in a subject's top-10%
positive edge list, check whether both endpoints share a Yeo-7 network
(within) or differ (between). Tally raw counts -- 7 within-network numbers
+ 21 between-network-pair numbers = 28 per subject -- sorted descending.

Uses the same `edges_for_subject`, `roi_network`, `NETWORK_NAMES`,
`_name_to_idx` already built in Section 5 above. Hub creation is untouched.

In [10]:
def compute_network_pair_counts(edges, roi_network, network_names, name_to_idx):
    """Count-based version (not density). For every edge, check if both
    endpoints share a network (within) or differ (between), tally raw counts.
    Returns 7 within-network counts + 21 between-network-pair counts,
    each dict sorted descending by count."""
    within = {net: 0 for net in network_names}
    skipped = 0
    between = {}
    for a_idx, net_a in enumerate(network_names):
        for net_b in network_names[a_idx + 1:]:
            between[f"{net_a}-{net_b}"] = 0

    for (name_i, w, name_j) in edges:
        net_i = roi_network[name_to_idx[name_i]]
        net_j = roi_network[name_to_idx[name_j]]
        if net_i not in network_names or net_j not in network_names:
            skipped += 1
            continue

        if net_i == net_j:
            within[net_i] += 1
        else:
            key = f"{net_i}-{net_j}" if f"{net_i}-{net_j}" in between else f"{net_j}-{net_i}"
            between[key] += 1

    within = dict(sorted(within.items(), key=lambda x: x[1], reverse=True))
    between = dict(sorted(between.items(), key=lambda x: x[1], reverse=True))
    return {"within_network": within, "between_network": between}


def run_community_counts(edge_list_names_all, roi_names, roi_network, network_names, name_to_idx):
    ground_truth = {}
    for sid in edge_list_names_all.keys():
        edges = edges_for_subject(sid)
        community_counts = compute_network_pair_counts(edges, roi_network, network_names, name_to_idx)
        gt = int(
            meta.loc[
                meta["subject_id"] == sid,
                "DX_binary"
            ].values[0]
        )

        ground_truth_class = "Control" if gt == 0 else "ADHD"

        # Keep the existing community counts and ADD classification
        ground_truth[sid] = {
            "ground_truth": ground_truth_class,
            "within_network": community_counts["within_network"],
            "between_network": community_counts["between_network"]
        }
    return ground_truth


community_ground_truth_counts = run_community_counts(
    edge_list_names_all, roi_names, roi_network, NETWORK_NAMES, _name_to_idx
)

with open("community_ground_truth_counts.json", "w") as f:
    json.dump(community_ground_truth_counts, f, indent=2)

print(f"Saved community_ground_truth_counts.json ({len(community_ground_truth_counts)} subjects)")

Saved community_ground_truth_counts.json (768 subjects)


In [11]:
import json
import os

# Define the file path
file_path = 'community_ground_truth_counts.json'

def check_community_counts(path):
    if not os.path.exists(path):
        print(f"Error: File not found at {path}")
        return

    with open(path, 'r') as f:
        data = json.load(f)

    all_valid = True
    mismatches = []

    print(f"Checking community structures in {path}...\n")

    for subject_id, content in data.items():
        # Get counts
        within_counts = len(content.get("within_network", {}))
        between_counts = len(content.get("between_network", {}))
        total = within_counts + between_counts
        
        # Validation logic (8 within, 28 between)
        if within_counts != 8 or between_counts != 28:
            all_valid = False
            mismatches.append({
                "id": subject_id,
                "within": within_counts,
                "between": between_counts,
                "total": total
            })

    if all_valid:
        print(f"✅ Success! All {len(data)} subjects have 8 within-network and 28 between-network entries (Total 36).")
    else:
        print(f"❌ Found {len(mismatches)} subjects with incorrect community counts:")
        for m in mismatches:
            print(f" - Subject {m['id']}: Within={m['within']}, Between={m['between']} (Total {m['total']})")

# Run the check
check_community_counts(file_path)

Checking community structures in community_ground_truth_counts.json...

✅ Success! All 768 subjects have 8 within-network and 28 between-network entries (Total 36).


# Top 3 Yeo-7 networks that the LLM considers most associated with ADHD

In [14]:
import os
import json
import re
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-8B" 
# MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"   
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"   # swapped to Llama -- gated, run huggingface-cli login first

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",       # spreads layers across available GPU(s), offloads to CPU if needed
    torch_dtype=torch.bfloat16,
)
model.eval()
print("Model loaded.")
# print(model.hf_device_map)


/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   1%|          | 2/291 [00:00<00:25, 11.19it/s]/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 291/291 [00:01<00:00, 177.10it/s]


Model loaded.


In [15]:
# -------------------------------------------------------------------
# ADHD Yeo-7 Prior-Knowledge Probe
# Ask the LLM for the top 3 Yeo-7 networks most strongly associated
# with ADHD based on its existing neuroscience knowledge.
# -------------------------------------------------------------------

MAX_NEW_TOKENS_COMMUNITY = 400
MAX_NEW_TOKENS_CLS = 1200

def build_adhd_network_knowledge_prompt():
    return """You are a neuroscience expert.

Among the seven Yeo functional networks:
Vis, SomMot, DorsAttn, SalVentAttn, Limbic, Cont, Default

Based ONLY on your existing neuroscience knowledge, identify the
TOP 3 networks most strongly associated with ADHD-related brain
functional connectivity differences.

Rank them from most to least associated with ADHD.

Do not use any subject-specific data or functional connectivity
information.

Output ONLY in this format:
[Network1, Network2, Network3]
"""

def generate_response_safe(prompt, system_message, max_new_tokens):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)

    del inputs, output_ids, generated
    torch.cuda.empty_cache()
    
    return text

def generate_adhd_network_knowledge_response():
    prompt = build_adhd_network_knowledge_prompt()

    system_message = (
        "You are a neuroscience expert. "
        "Output only the requested bracketed list. "
        "No explanation, no markdown, and no extra text."
    )

    return generate_response_safe(
        prompt,
        system_message,
        MAX_NEW_TOKENS_COMMUNITY
    )

In [16]:
adhd_network_response = generate_adhd_network_knowledge_response()
print("The top 3 Yeo-7 networks that the LLM considers most associated with ADHD are: " + adhd_network_response)

/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


The top 3 Yeo-7 networks that the LLM considers most associated with ADHD are: [Default, Limbic, DorsAttn]
